<a href="https://colab.research.google.com/github/TinoTonic/REDCap-SDTM-automation/blob/feat%2Fimprove_fuzzy_matching/Gen_SDTM_Map.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Required Input Files

The notebook requires the following files:

- REDCap data dictionary in .csv

- SDTM controlled terminology (`SDTM Terminology 2026-03-27`)

- SDTMIG (`SDTMIG_v3.4.xlsx`)

If you want to generate an annotated CRF:

- Annotated CRF PDF generated by the external module `Annotated PDF`.



## 1. Installing and importing packages

In [19]:
pip install rapidfuzz pymupdf

In [20]:
import re
import pandas as pd
import pymupdf as fitz
from rapidfuzz import process, fuzz

## 2. Reading the Data Dictionary and SDTM files

The REDCap Data Dictionary is loaded from the `redcap_data.csv` file.

Columns that are not required for the SDTM mapping pipeline are removed from the original DataFrame to simplify subsequent processing.




In [12]:
redcap_data = pd.read_csv("redcap_data.csv")

redcap_data = redcap_data.rename(columns={
    "Variable / Field Name": "Field Name",
    "Choices, Calculations, OR Slider Labels": "Choices"
     })
redcap_data = redcap_data.drop(columns=[
    'Question Number (surveys only)',
    'Section Header',
    'Field Note',
    'Text Validation Min',
    'Text Validation Max',
    'Branching Logic (Show field only if...)',
    'Custom Alignment',
    'Question Number (surveys only)',
    'Matrix Group Name',
    'Matrix Ranking?'
    ])

terminology = pd.read_excel("SDTM Terminology.xls", sheet_name="SDTM Terminology 2026-03-27")

sdtm_variables = pd.read_excel("SDTMIG_v3.4.xlsx", sheet_name = "Variables")

sdtm_dataset = pd.read_excel("SDTMIG_v3.4.xlsx", sheet_name = "Datasets")


## 3. Reading REDCap Field Annotations

The code below uses two regular expressions to parse metadata from REDCap.

- **`sdtm_pattern`** parses SDTM annotations written as:
  - `SDTM:IT.domain.variable.testcd;`
  - Captured groups:
    - `domain`
    - `variable`
    - `testcd` (optional)

- `[.,:]` allows `.`, `,`, or `:` as separators to tolerate minor typing inconsistencies in the annotations.


- **`submission_pattern`** parses REDCap submission options from the Data Dictionary, such as:
  - `Y, Yes | N, No`
  - Captured groups:
    - `REDCap_Value` = `Y` | `N`
    - `REDCap_Term` = `Yes` | `No`




In [13]:
redcap_values = []
domains = []

sdtm_pattern = r"SDTM[.,:]IT[.,:](?P<DOMAIN>\w+)\.(?P<SDTM_VARIABLE>\w+)(?:\.(?P<TESTCD>\w+))?[;.:]?"
submission_pattern = r"\s*(?P<REDCap_Value>[^,|]+)\s*,\s*(?P<REDCap_Term>[^|]+)"

for _, row in redcap_data.iterrows():
    field_annotation = str(row["Field Annotation"]).strip()
    field_name = str(row["Field Name"]).strip()
    values = str(row["Choices"]).strip()
    field_type = str(row["Field Type"]).strip()
    if field_type == "sql":
        continue
    if field_name.startswith("desc_"):
        redcap_data.at[_, "Field Label"] = ""
    match = re.search(sdtm_pattern, field_annotation)
    if match:
        redcap_data.at[_, "DOMAIN"] = match.group("DOMAIN")
        redcap_data.at[_, "SDTM VARIABLE"] = match.group("SDTM_VARIABLE")
        redcap_data.at[_, "TESTCD"] = match.group("TESTCD")
    else:
        redcap_data.at[_, "DOMAIN"] = ""
        redcap_data.at[_, "SDTM VARIABLE"] = ""
        redcap_data.at[_, "TESTCD"] = ""
    if redcap_data.at[_, "DOMAIN"] != "":
        domains.append({
        "Domain": redcap_data.at[_, "DOMAIN"]
        })

    for match in re.finditer(submission_pattern, values):
          redcap_values.append({
              "Field Name": field_name,
              "REDCap Value": match.group("REDCap_Value"),
              "REDCap Term": match.group("REDCap_Term").strip()
          })

Variable_Metadata = redcap_data.copy()

REDCap_Values = pd.DataFrame(redcap_values)

## 4. Creating the Dataset Metadata File

This step creates a separate Excel file containing the **`Datasets`** worksheet.

The worksheet lists all SDTM domains identified in the CRF, together with their corresponding class, description, and structure.

- **`supp_list`** identifies all SUPPQUAL domains present in the CRF and stores them in a separate list.

- **`datasetmetadata`** merges the identified domains (generated in the previous code cell) with the **SDTMIG v3.4** metadata to retrieve the corresponding class, description, and structure.


Since **SDTMIG v3.4** does not include metadata for all **SUPP--** domains, the script uses **`supp_list`** to assign the default SUPPQUAL structure to those datasets.



In [14]:
domains = pd.DataFrame(domains)
domains = domains.drop_duplicates()
supp_list = [dom[-2:] for dom in domains["Domain"] if dom.startswith("SUPP")]

datasetmetadata = (domains.merge(
        sdtm_dataset,
        left_on="Domain",
        right_on="Dataset Name",
        how="inner"
        )
        [["Dataset Name","Class","Dataset Label","Structure"]]
)

supp_rows_data = []

for dom in supp_list:
    new_row = {
        "Dataset Name": f"SUPP{dom}",
        "Class": "Relationship",
        "Dataset Label": f"Supplemental Qualifiers for {dom}",
        "Structure": "One record per IDVAR, IDVARVAL, and QNAM value per subject",
        }
    supp_rows_data.append(new_row)

df_supp_metadata = pd.DataFrame(supp_rows_data)

Dataset_Metadata = pd.concat([datasetmetadata, df_supp_metadata], ignore_index=True)

## 4. Reading Files

The data generated in the previous steps is loaded into working Data Frames.

- **`mapping`** contains the variable metadata extracted from the REDCap Data Dictionary.
- **`codelist`** contains the REDCap submission values and their corresponding terms.

Missing values are replaced with empty strings to simplify subsequent processing.



In [15]:
mapping = Variable_Metadata.copy()
mapping.fillna("")

codelist = REDCap_Values.copy()
codelist.fillna("")

,Field Name,REDCap Value,REDCap Term
0,ie_ieyn_c1,N,No
1,ie_ieyn_c1,Y,Yes
2,ie_iecat_c1,INCLUSION,Inclusion
3,ie_iecat_c1,EXCLUSION,Exclusion
4,ie_inc_ietestcd_c1,INCL01,Inclusion 1
...,...,...,...
5035,ds_eos_dsdecod_c60,8,Screen Failure
5036,ds_eos_dsdecod_c60,9,Site Terminated by Sponsor
5037,ds_eos_dsdecod_c60,10,Study Terminated By Sponsor
5038,ds_eos_dsdecod_c60,11,Withdrawal by Subject


## 5. Filtering and Generating values of Mapped Variables

This step creates a separate sheet that merges REDCap Submission Values and SDTM Variables of every field.

- Variables without an SDTM mapping are removed.
- The SDTM variable **`RACE`** is renamed to **`RACEC`** to match the SDTM standard.
- The filtered variable metadata is merged with the REDCap submission values using the REDCap field name.

The resulting table contains, for each mapped variable, its SDTM variable name, optional `TESTCD`, and the corresponding REDCap values and terms. This table is used to generate the SDTM codelists.


In [16]:
mapping["SDTM VARIABLE"] = (mapping["SDTM VARIABLE"].astype(str).str.strip())

mapping_filtered = mapping[(mapping["SDTM VARIABLE"] != "") & (mapping["SDTM VARIABLE"] != "nan")]

mapping_filtered = pd.DataFrame(mapping_filtered)

mapping_filtered.loc[mapping_filtered["SDTM VARIABLE"] == "RACE", "SDTM VARIABLE"] = "RACEC"

Codelists = (mapping_filtered.merge(
        codelist,
        left_on="Field Name",
        right_on="Field Name",
        how="inner"
        )
        [["Field Name","SDTM VARIABLE","TESTCD","REDCap Term","REDCap Value"]]
)

CRF_SDTM_Variables = Codelists.to_dict("records")
CRF_SDTM_Variables = pd.DataFrame(CRF_SDTM_Variables)

## 6. Linking Mapped Variables to CDISC Codelists

This step links each mapped SDTM variable to its corresponding CDISC controlled terminology.

- The script searches the SDTM variable metadata to identify the **CDISC CT Codelist Code** associated with each mapped variable.
- Using the codelist code, it retrieves all controlled terminology terms from the CDISC terminology file.
- For variables that are themselves represented as CDISC submission values (e.g., `RACE` or `RACEC`), the script also identifies and links their corresponding codelists.
- Duplicate entries are removed to ensure that each variable -> codelist relationship is stored only once.

The resulting **`Codelist_Codes`** Data Frame contains, for each mapped variable, its associated CDISC codelist, terminology codes, submission values, and preferred terms. This information is used in the subsequent generation of SDTM codelists.


In [17]:
link = {}
linked_by_code = []

for _, row in mapping.iterrows():
    field_name = str(row["Field Name"]).strip()
    domain = str(row["DOMAIN"]).strip()
    variable = str(row["SDTM VARIABLE"]).strip()
    testcd = str(row["TESTCD"]).strip()
    if not variable:
        continue

    for _, row in sdtm_variables.iterrows():
        var = str(row["Variable Name"]).strip()
        if var == variable:
            codelistcode = str(row["CDISC CT Codelist Code(s)"]).strip()
            if codelistcode == "nan":
                continue
            link[variable] = {"Variable": variable,"Codelist Code": codelistcode}

df = pd.DataFrame(link).T

for _, df_row in df.iterrows():
    variable = df_row["Variable"]
    dfclcode = df_row["Codelist Code"]

    for _, term_row in terminology.iterrows():
        clcode = str(term_row["Codelist Code"]).strip()
        subval = str(term_row["CDISC Submission Value"]).strip()

        if clcode == dfclcode:
            linked_by_code.append({
                "Variable": variable,
                "Codelist Name": str(term_row["Codelist Name"]).strip(),
                "Codelist Code": dfclcode,
                "Codes": str(term_row["Code"]).strip(),
                "Submission Value": str(term_row["CDISC Submission Value"]).strip(),
                "NCI Pref. Term": str(term_row["NCI Preferred Term"]).strip()
            })
        if subval == variable or subval == f"{variable}C":
            code = str(term_row["Code"]).strip()
            saved_vars = [item["Variable"] for item in linked_by_code]
            for _, term_row in terminology.iterrows():
                if code == str(term_row["Codelist Code"]).strip():
                    if subval not in saved_vars:
                        linked_by_code.append({
                            "Variable": subval,
                            "Codelist Name": str(term_row["Codelist Name"]).strip(),
                            "Codelist Code": str(term_row["Codelist Code"]).strip(),
                            "Codes": str(term_row["Code"]).strip(),
                            "Submission Value": str(term_row["CDISC Submission Value"]).strip(),
                            "NCI Pref. Term": str(term_row["NCI Preferred Term"]).strip()
                        })

Codelist_Codes = pd.DataFrame(linked_by_code)
Codelist_Codes = Codelist_Codes.drop_duplicates()

## 7. Generating Study Specific Controlled Terminology Translation

This step generates the study specific controlled terminology by mapping REDCap terms to their corresponding SDTM controlled terminology values.

- The available SDTM submission values are extracted from the previously generated **`Codelist_Codes`** DataFrame.
- A fuzzy matching approach is used to identify the closest SDTM submission value for each REDCap term. This allows minor differences in wording between the CRF and CDISC terminology to be matched.
- REDCap submission `**terms** are used for matching instead of REDCap values. This is because REDCap values are often stored as coded responses, for example:

  `1, Black | 2, White | 3, Asian`

  Matching based on the terms allows the CRF labels to be updated to better align with CDISC preferred terminology without affecting the stored values or existing branching logic.


- Only matches above the defined similarity threshold are accepted.
- Specific SDTM variable names are standardized to ensure consistency with SDTM controlled terminology:
  - `QVAL` variables are renamed using their corresponding `TESTCD`.
  - `RACE` is mapped to `RACEC`.
  - `ETHNIC` and `CETHNIC` are mapped to `ETHNICC`.

The matched REDCap terms are then merged with the CDISC codelist metadata to create the **`Controlled_Terminology`** DataFrame.

The final output contains:
- REDCap field information
- SDTM variable and test code
- CDISC codelist name and code
- Controlled terminology codes
- REDCap values and terms
- Corresponding CDISC submission values and NCI preferred terms

Duplicate mappings are removed to ensure that each SDTM variable, test code, codelist, and terminology code combination is uniquely represented.


In [25]:
SDTM_Values = Codelist_Codes["Submission Value"].astype(str).tolist()
NCIPref_Terms = Codelist_Codes["NCI Pref. Term"].astype(str).tolist()

def get_fuzz_match(text, choices_list, threshold=50):
    result = process.extractOne(str(text), choices_list)
    if result is None:
        return None, 0

    match, score, _ = result

    if score >= threshold:
        return match, score
    else:
        return None, 0

    if score >= threshold:
        return match, score
    else:
        return None, 0

def get_best_match(text):
    sdtm_match, sdtm_score = get_fuzz_match(text, SDTM_Values, threshold=70)
    nci_match, nci_score = get_fuzz_match(text, NCIPref_Terms, threshold=70)

    if sdtm_score >= nci_score:
        return sdtm_match
    else:
        return Codelist_Codes.loc[
                Codelist_Codes["NCI Pref. Term"] == nci_match,"Submission Value"
                ].iloc[0]

CRF_SDTM_Variables["Temp"] = CRF_SDTM_Variables["REDCap Term"].apply(
    get_best_match)

CRF_SDTM_Variables.loc[
    CRF_SDTM_Variables["SDTM VARIABLE"] == "QVAL", "SDTM VARIABLE"
    ] = CRF_SDTM_Variables["TESTCD"]
CRF_SDTM_Variables.loc[
    CRF_SDTM_Variables["SDTM VARIABLE"] == "RACE", "SDTM VARIABLE"
    ] = "RACEC"
CRF_SDTM_Variables.loc[
    CRF_SDTM_Variables["SDTM VARIABLE"] == "ETHNIC", "SDTM VARIABLE"
    ] = "ETHNICC"
CRF_SDTM_Variables.loc[
    CRF_SDTM_Variables["SDTM VARIABLE"] == "CETHNIC", "SDTM VARIABLE"
    ] = "ETHNICC"

Controlled_Terminology = (
    CRF_SDTM_Variables.merge(
        Codelist_Codes,
        left_on=["SDTM VARIABLE","Temp"],
        right_on=["Variable","Submission Value"],
        how="right"
    )
    [["Field Name","SDTM VARIABLE","TESTCD","Codelist Name","Codelist Code", "Codes", "REDCap Value", "REDCap Term", "Submission Value", "NCI Pref. Term"]]
)

Controlled_Terminology.loc[Controlled_Terminology["SDTM VARIABLE"] == "RACEC", "SDTM VARIABLE"] = "RACE"

CRF_SDTM_Variables.drop(columns=["Temp"], inplace=True)

Controlled_Terminology = Controlled_Terminology.drop_duplicates(subset=["SDTM VARIABLE", "TESTCD", "Codelist Code", "Codes"])

## 8. Output Files

In [10]:
with pd.ExcelWriter("REDCap Mapping Spec.xlsx", engine="openpyxl") as writer:

    Dataset_Metadata.to_excel(writer, sheet_name="Dataset Metadata", index=False)

    Variable_Metadata.to_excel(writer, sheet_name="Variable Metadata", index=False)

    REDCap_Values.to_excel(writer, sheet_name="REDCap Values", index=False)

    Codelist_Codes.to_excel(writer, sheet_name="Variables Codelists", index=False)

    Controlled_Terminology.to_excel(writer, sheet_name="Controlled Terminology", index=False)